In [ ]:
#@title Installing required packages

!pip install deepface
!pip install retina-face

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 2.8 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.5.0-py2.py3-none-any.whl size=116934 sha256=0d1f5317f4f993025721c6e33ba58c72c36518c12deac918cb3425a0ce5fbc55
  Stored in directory: /root/.cache/pip/wheels/90/d4/f7/9404e5db0116bd4d43e5666eaa3e70ab53723e1e3ea40c9a95
Successfully built fire


In [ ]:
#@title Import Packages

# Data handling
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Image specific
import cv2
from google.colab.patches import cv2_imshow

# RetinaFace
from retinaface import RetinaFace
from retinaface.commons import preprocess, postprocess
# DeepFace
from deepface import DeepFace

import uuid
from tqdm.notebook import tqdm

import os
from os import path
import warnings
import pickle

24-01-25 10:27:19 - Directory /root/.deepface created
24-01-25 10:27:19 - Directory /root/.deepface/weights created


In [ ]:
#@title Load an unzip images for face detection

!unzip -q ../data/insta-story-images.zip -d ./images
!unzip -q ../data/2024-01-03-database-final-v4.zip -d ./database_final-v1

In [ ]:
#@title Import data farmes

all_faces_v2_df = pd.read_csv('../data/AllFaces_v2.csv')
images_v2_df = pd.read_csv('../data/images_v2.csv')
all_faces_tasks_df = pd.read_csv('../data/AllFaces-Tasks.csv')

In [ ]:
#@title Drop NaN's
#@markdown Drop all rows without any face found

images_v2_df.dropna(inplace = True)

In [ ]:
#@title ## Deal with list as values

images_v2_df['face_ids'] = images_v2_df["face_ids"].apply(eval)

all_faces_v2_df["facial_area"] = all_faces_v2_df["facial_area"].apply(eval)
all_faces_v2_df["right_eye"] = all_faces_v2_df["right_eye"].apply(eval)
all_faces_v2_df["left_eye"] = all_faces_v2_df["left_eye"].apply(eval)
all_faces_v2_df["nose"] = all_faces_v2_df["nose"].apply(eval)
all_faces_v2_df["mouth_right"] = all_faces_v2_df["mouth_right"].apply(eval)
all_faces_v2_df["mouth_left"] = all_faces_v2_df["mouth_left"].apply(eval)

## Preparing Faces

In [ ]:
#@title Align-pre detected faces

def extract_faces_adapted(filepath, align = True, resize = False):
  """
  This functions builds upon the extract_faces function from Sefik Serengils RetinaFace
  https://github.com/serengil/retinaface/blob/master/retinaface/RetinaFace.py#L58.

  extract_faces_adapted extracts and aligns (optinal) all faces in one image.

  Parameters:
      filepath (string): exact filepath of the image, where all face should be
                         extracted from

      align (boolean): toggle face alignment (default is True)

  Returns:
      Returns a list of dicts. Each dict has one key, value pair with an id and
      a corresponding extracted face image.
  """

  responses = []

  image = cv2.imread(filepath)

  current_image = images_v2_df.loc[images_v2_df['filepath'] == filepath]
  faces = current_image['face_ids'].item()

  for id in faces:
    current_face = all_faces_v2_df.loc[all_faces_v2_df['uuid'] == id]

    facial_area = current_face['facial_area'].item()
    facial_img = image[facial_area[1]: facial_area[3], facial_area[0]: facial_area[2]]

    if align == True:
      right_eye = current_face['right_eye'].item()
      left_eye =  current_face['left_eye'].item()
      nose = current_face['nose'].item()
      facial_img = postprocess.alignment_procedure(facial_img, right_eye, left_eye, nose)

    # if resize:
    #   facial_img = resize_db_face(facial_img)

    response = {id: facial_img[:, :, ::1]}
    responses.append(response)

  return responses

In [ ]:
if os.path.exists('/content/Exports') == False:
  os.mkdir('/content/Exports')


for image in tqdm(images_v2_df['filepath']):
  all_current_faces = extract_faces_adapted(image)

  for dic in all_current_faces:
    for key in dic:
      file_name = '/content/Exports/{}.jpg'.format(key)
      cv2.imwrite(file_name, dic[key])

  0%|          | 0/1787 [00:00<?, ?it/s]

In [ ]:
!zip -r faces-extracted Exports

Die letzten 5000 Zeilen der Streamingausgabe wurden abgeschnitten.
  adding: Exports/3310e3ec-cb95-4695-8d65-12894c28edba.jpg (deflated 24%)
  adding: Exports/9c0fa8d7-848d-4233-9366-8bc51a819ee1.jpg (deflated 12%)
  adding: Exports/f8426454-5843-4407-9330-bc688edf335b.jpg (deflated 21%)
  adding: Exports/bfbfce21-a1c1-401a-9e96-87ee88c2ebac.jpg (deflated 13%)
  adding: Exports/d1abdd54-a6fe-420d-a6a2-546c6badb447.jpg (deflated 24%)
  adding: Exports/e3c6115b-9b48-471a-b285-6c51875c223b.jpg (deflated 27%)
  adding: Exports/e9450095-f3c1-43ee-b0ca-6a9a884431a2.jpg (deflated 10%)
  adding: Exports/3a92b357-d0ec-48c5-9552-0bd4d7f8dfbd.jpg (deflated 15%)
  adding: Exports/1213fa4d-a4d6-414d-b673-49140f38be39.jpg (deflated 17%)
  adding: Exports/262dcee0-4da4-47dc-afea-6f11c36b7048.jpg (deflated 20%)
  adding: Exports/89ee1448-aa84-4515-b5cc-fd1a16cb8e8b.jpg (deflated 2%)
  adding: Exports/063ca103-e673-4f64-bf14-7d78ae6d3f4b.jpg (deflated 2%)
  adding: Exports/34d85260-e47f-4ea8-9fe3-223ba

## Preparing Face Database
Hier habe ich mich nun auf die Kanzlerkandidaten konzentriert.

QuickFix um mit Umlauten gut klar zu kommen.

In [ ]:
import os
import unicodedata

def replace_characters(original):
    normalized = unicodedata.normalize('NFC', original)
    # Replace specific characters in the string
    return normalized.replace('ä', 'ae').replace('ö', 'oe').replace('ü', 'ue')

def rename_files_and_folders(path):
    for root, dirs, files in os.walk(path, topdown=False):
        # Rename files
        for name in files:
            new_name = replace_characters(name)
            if new_name != name:
                os.rename(os.path.join(root, name), os.path.join(root, new_name))

        # Rename directories
        for name in dirs:
            new_name = replace_characters(name)
            if new_name != name:
                os.rename(os.path.join(root, name), os.path.join(root, new_name))

rename_files_and_folders(DATABASE)

Filtere nach den Accounts, die mich interessieren

In [ ]:
# Filtere für Kanzlerkandidaten
selected_accounts = ['armin_laschet', 'abaerbock', 'christlichsozialeunion',
       'christianlindner', 'cdu', 'fdp', 'die_gruenen', 'markus.soeder',
       'spdde', 'olafscholz']

all_faces_v2_df = all_faces_v2_df[all_faces_v2_df['account_name'].isin(selected_accounts)]

In [ ]:
all_faces_v2_df.head()

Mapping Account <-> Partei

In [ ]:
party_dict = {
  "armin_laschet": "CDU",
  "spdde": "SPD",
  "afd.bund": "AfD",
  "nicola_beer": "FDP",
  "christianlindner": "FDP",
  "abaerbock": "GRUEN",
  "cdu": "CDU",
  "robert.habeck":  "GRUEN",
  "olafscholz": "SPD",
  "fw_bayern": "FW",
  "die_gruenen":  "GRUEN",
  "christlichsozialeunion":  "CSU",
  "fdp": "FDP",
  "saskiaesken":  "SPD",
  "susanne_hennig_wellsow": "Linke",
  "atesgurpinar":  "Linke",
  "grey_gor": "FW",
  "markus.soeder":  "CSU",
  "dielinke":  "Linke",
  "joerg.meuthen": "AfD",
  "engin_eroglu_": "FW",
}
all_faces_v2_df['Party'] = all_faces_v2_df['account_name'].map(party_dict)

In [ ]:
all_faces_v2_df.head()

Schiebe die Bilder in einen neuen Unterordner ( `Datenbank > Partei > Name `)

In [ ]:
import os
import shutil

accounts_dict = {
  "CDU": ["Armin Laschet"],
  "CSU": ["Markus Soeder"],
  "SPD": ["Olaf Scholz"],
  "FDP": ["Christian Lindner"],
  "GRUEN": ["Annalena Baerbock"],
}

def copy_directory(src, dst):
    if os.path.exists(dst):
        os.rename(dst, dst + '_backup')
    shutil.copytree(src, dst)

# New base directory for the structured data
new_base_dir = "database_v2"

# Iterate through parties and their members in the dictionary
for party, names in accounts_dict.items():
    for name in names:
        # Source and destination directory paths
        src_dir = os.path.join(DATABASE, name)
        dst_dir = os.path.join(new_base_dir, party, name)

        # Copy directory if it exists in the source
        if os.path.isdir(src_dir):
            copy_directory(src_dir, dst_dir)
        else:
            print(f"Directory not found: {src_dir}")

## Starte die Gesichtserkennung.
Die Datenbank wird beim ersten Mal für jede Partei generiert. Das erhöht die Geschwindigkeit.

In [ ]:
#@title Perform Face Recognition

DATABASE = "/content/database_v2"
FORCE_DETECT = False
BACKEND = "retinaface" #@param ['skip', 'opencv', 'ssd', 'dlib','mtcnn', 'retinaface', 'mediapipe', 'yolov8',' yunet']

def find_matches(image, model, metric, account_name):
    # Function to find matches for a given image using specified model and metric
    path = f'/content/Exports/{image}.jpg'
    col = f"{model}_{metric}"
    db_path = os.path.join(DATABASE, account_name)  # Updated db_path
    scores = DeepFace.find(img_path=path, db_path=db_path, model_name=model,
                           distance_metric=metric, detector_backend=BACKEND,
                           enforce_detection=FORCE_DETECT, silent=True)

    return scores, col


def process_scores(scores, col, image):
    # Process and return match details
    face_score, match_file, matches = np.nan, "", []
    if len(scores[0]) > 0:
        scores_df = scores[0]
        min_score = scores_df[scores_df[col] == scores_df[col].min()]
        face_score = min_score[col].iloc[0]
        match_file = min_score['identity'].iloc[0].split("/")[5]
        matches = [{"match_uuid": f'match_{uuid.uuid4()}', "face_uuid": image,
                    "match_file": row['identity'].split("/")[5],
                    "match_score": row[col], "model": model,
                    "distance_metric": metric}
                   for index, row in scores_df.iterrows()]
    return face_score, match_file, matches


# Main loop
model = "Facenet512"
metrics = ['euclidean_l2'] #'euclidean', 'cosine']

for metric in tqdm(metrics, desc='Processing Metrics'):
    faces, all_matches = [], []

    # Group by 'account_name' and iterate through each group
    for party, group in tqdm(all_faces_v2_df.groupby('Party'), desc='Processing parties'):
        for image in tqdm(group['uuid']):
            scores, col = find_matches(image, model, metric, party)
            face_score, match_file, matches = process_scores(scores, col, image)
            all_matches.extend(matches)
            if face_score is not np.nan:
                faces.append({"face_uuid": image, "match_file": match_file,
                              "match_score": face_score, "model": model,
                              "distance_metric": metric, "number_of_hits": len(scores[0])})

    # Creating and Saving DataFrames for each model-metric pair
    all_matches_df = pd.DataFrame.from_dict(all_matches)
    face_recognition_df = pd.DataFrame.from_dict(faces)

    all_matches_df.to_csv(f"{EXPORT_FOLDER}all_matches_{model}_{metric}_{VERSION}.csv", index=False)
    face_recognition_df.to_csv(f"{EXPORT_FOLDER}top_matches_{model}_{metric}_{VERSION}.csv", index=False)